# 10 - GCN Hyperparameter Tuning

**Bayesian optimization for GCN hyperparameters using Optuna.**

Based on paper methodology:
- 32 optimization trials
- Tree-structured Parzen Estimator (TPE) algorithm
- Early stopping with patience of 30 epochs
- Optimization metric: Validation ROC-AUC

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import os
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check for Optuna
try:
    import optuna
    from optuna.samplers import TPESampler
    print("✓ Optuna available")
except ImportError:
    print("⚠️ Install optuna: pip install optuna")

# PyTorch
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ============================================================================
# HYPERPARAMETER SEARCH SPACE
# ============================================================================
# Define the search space for GCN hyperparameters
SEARCH_SPACE = {
    'num_layers': (2, 4),           # Number of GCN layers
    'hidden_dim1': [32, 64, 128],   # First hidden dimension
    'hidden_dim2': [64, 128, 256],  # Second hidden dimension  
    'hidden_dim3': [128, 256, 512], # Third hidden dimension
    'classifier_hidden': [64, 128, 256],
    'dropout': (0.1, 0.5),          # Dropout rate (continuous)
    'learning_rate': (1e-4, 1e-2),  # Learning rate (log scale)
    'weight_decay': (1e-6, 1e-3)    # L2 regularization (log scale)
}

# Optimization settings
N_TRIALS = 32          # Number of optimization trials (paper: 32)
PATIENCE = 30          # Early stopping patience (paper: 30)
MAX_EPOCHS = 100       # Max training epochs

print("Search Space:")
for param, values in SEARCH_SPACE.items():
    print(f"  {param}: {values}")

In [ ]:
# ============================================================================
# OBJECTIVE FUNCTION
# ============================================================================
def objective(trial):
    """
    Optuna objective function for GCN hyperparameter optimization.
    Returns validation ROC-AUC (to be maximized).
    """
    # Sample hyperparameters
    num_layers = trial.suggest_int('num_layers', 2, 4)
    hidden_dim1 = trial.suggest_categorical('hidden_dim1', [32, 64, 128])
    hidden_dim2 = trial.suggest_categorical('hidden_dim2', [64, 128, 256])
    hidden_dim3 = trial.suggest_categorical('hidden_dim3', [128, 256, 512])
    classifier_hidden = trial.suggest_categorical('classifier_hidden', [64, 128, 256])
    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    
    # Build hidden dimensions based on num_layers
    if num_layers == 2:
        hidden_dims = [hidden_dim1, hidden_dim2]
    elif num_layers == 3:
        hidden_dims = [hidden_dim1, hidden_dim2, hidden_dim3]
    else:
        hidden_dims = [hidden_dim1, hidden_dim2, hidden_dim3, 256]
    
    print(f"\nTrial {trial.number}: layers={num_layers}, dims={hidden_dims}, lr={learning_rate:.6f}")
    
    # Here you would:
    # 1. Create model with sampled hyperparameters
    # 2. Train model with early stopping
    # 3. Return best validation ROC-AUC
    
    # Placeholder return (replace with actual training)
    val_auc = np.random.uniform(0.5, 0.8)  # Simulated
    return val_auc

print("✓ Objective function defined")

In [ ]:
# ============================================================================
# RUN OPTIMIZATION (example with 3 trials)
# ============================================================================
# Create study with TPE sampler (Bayesian optimization)
study = optuna.create_study(
    direction='maximize',  # Maximize ROC-AUC
    sampler=TPESampler(seed=42),
    study_name='GCN_hyperopt'
)

# Run a few trial as demonstration (use N_TRIALS=32 for full optimization)
print("Running demonstration optimization (3 trials)...")
study.optimize(objective, n_trials=3, show_progress_bar=True)

# Results
print(f"\n{'='*60}")
print("OPTIMIZATION RESULTS")
print(f"{'='*60}")
print(f"Best trial: #{study.best_trial.number}")
print(f"Best validation ROC-AUC: {study.best_value:.4f}")
print(f"\nBest hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

In [ ]:
# ============================================================================
# VISUALIZE OPTIMIZATION HISTORY
# ============================================================================
import matplotlib.pyplot as plt

# Plot optimization history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trial values
trials = [t.number for t in study.trials]
values = [t.value for t in study.trials]
axes[0].plot(trials, values, 'o-')
axes[0].axhline(y=study.best_value, color='r', linestyle='--', label='Best')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Validation ROC-AUC')
axes[0].set_title('Optimization History')
axes[0].legend()

# Parameter importance (if enough trials)
if len(study.trials) >= 3:
    importance = optuna.importance.get_param_importances(study)
    params = list(importance.keys())[:5]  # Top 5
    values = [importance[p] for p in params]
    axes[1].barh(params, values)
    axes[1].set_xlabel('Importance')
    axes[1].set_title('Top Hyperparameter Importance')

plt.tight_layout()
plt.show()

print("\n✓ Hyperparameter tuning complete!")